<a href="https://colab.research.google.com/github/mSharifHub/vision-transformer/blob/main/vision_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
import torchvision
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.utils.data as dataloader
import torch.nn as nn

In [11]:
transformation = transforms.Compose([transforms.ToTensor()])

In [12]:
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transformation)
test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transformation)

In [17]:
batch_size = 64
num_classes = 10
num_channels = 1
img_size = 28
patch_size = 7  # 28 divisible by 7.
patch_num = (img_size/ patch_size) * (img_size/ patch_size)
attention_heads = 4
embed_dimension = 16
transformers_blocks = 4
mlp_size = embed_dimension * 4  # T MLP are typically 4x the embed dimension ( Original Transformers Paper )

In [14]:
train_data_loader = dataloader.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_data_loader = dataloader.DataLoader(test_data, batch_size=batch_size, shuffle=True)

In [24]:
class PatchEmbedding(nn.Module):

  def __init__(self):
    super().__init__()
    self.embed = nn.Conv2d(num_channels, embed_dimension, kernel_size=patch_size, stride=patch_size)


  def forward(self, x):
    return self.embed(x).flatten(2).transpose(1, 2)


In [25]:
class TransformerEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_normalization1 = nn.LayerNorm(embed_dimension)
    self.layer_normalization2= nn.LayerNorm(embed_dimension)
    self.attention = nn.MultiheadAttention(embed_dim=embed_dimension, num_heads=attention_heads)
    self.mlp = nn.Sequential(
        nn.Linear(embed_dimension, mlp_size),
        nn.ReLU(),
        nn.Linear(mlp_size, embed_dimension)
    )

In [22]:
images, label = next(iter(train_data_loader))
patch_embed = nn.Conv2d(num_channels, embed_dimension, kernel_size=patch_size, stride=patch_size)
print(images.shape)

print(patch_embed(images).shape)

print(patch_embed(images).flatten(2).shape)
print(patch_embed(images).flatten(2).transpose(1, 2).shape)


torch.Size([64, 1, 28, 28])
torch.Size([64, 16, 4, 4])
torch.Size([64, 16, 16])
torch.Size([64, 16, 16])
